# Study v2 training plan — path-corrected copy (DO NOT RUN)

This copy is staged for review only. The training batch loops were not executed.

- ROOT: /home/sagemaker-user/brain-mri-segmentation
- VERSIONDIR and subprocess cwd: ROOT/versions/study_v2
- PYTHONPATH: VERSIONDIR/src
- train/val data root: ROOT/data (existing train/val only)
- output root: ROOT/runs/study_v2
- planned grid: 6 main + 6 diagnostic + 2 sensitivity = 14 runs; seeds 0, 1, 2.


# 03 — Planned study v2 (14 runs). Run ONLY after: 50-ep baseline finished, proposal finalized, staged fixes applied + 28 tests pass.
Holdout is not staged or read here. Each cell skips completed runs (run_meta.json present).

In [ ]:
import json, os, subprocess, sys
ROOT=os.path.abspath('/home/sagemaker-user/brain-mri-segmentation'); VERSIONDIR=os.path.join(ROOT,'versions','study_v2'); DATA=os.path.join(ROOT,'data'); OUT=os.path.join(ROOT,'runs','study_v2'); ENV=os.environ.copy(); ENV['PYTHONPATH']=os.path.join(VERSIONDIR,'src')
M=json.load(open(f'{VERSIONDIR}/docs/study_v2_execution_manifest.json'))
assert not os.path.exists(f'{DATA}/test') and not os.path.exists(f'{DATA}/test_manifest.csv'), 'holdout must not be staged before freeze'
subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=VERSIONDIR,env=ENV,check=True)
def run(c,s):
    rd=f'{OUT}/{c}_s{s}'
    if os.path.exists(f'{rd}/run_meta.json'): print('skip done',rd); return
    if os.path.exists(rd): raise SystemExit(f'{rd} partial: move it aside and log it before retrying')
    subprocess.run([sys.executable,'scripts/train.py','--config',f'configs/{c}.yaml','--data-root',DATA,'--out-root',OUT,'--seed',str(s)],cwd=VERSIONDIR,env=ENV,check=True)
PLAN=[(r['config'].split('/')[-1][:-5],r['seed']) for r in M['runs']]; print(len(PLAN),PLAN)

In [ ]:
# MAIN: bce_baseline vs bce_dice, seeds 0,1,2 (6 runs)
# for c,s in PLAN[:6]: run(c,s)

In [ ]:
# DIAGNOSTIC: weighted_bce, bce_dice_posonly, seeds 0,1,2 (6 runs)
# for c,s in PLAN[6:12]: run(c,s)

In [ ]:
# SENSITIVITY: dice weight 0.5 and 2, seed 0 (2 runs)
# for c,s in PLAN[12:]: run(c,s)

In [ ]:
# VAL-ONLY summary table (no test)
import pandas as pd
rows=[]
for c,s in PLAN:
    p=f'{OUT}/{c}_s{s}/run_meta.json'
    if os.path.exists(p):
        m=json.load(open(p)); v=m['val_at_selected']; rows.append(dict(cond=c,seed=s,thr=m['selected_threshold'],best_epoch=m.get('best_epoch'),pooled_dice=v['pooled_foreground_dice'],iou=v['pooled_foreground_iou'],pos_dice=v['positive_slice_dice'],empty_fp=v['empty_slices_with_fp'],fp_px=v['empty_fp_pixels_total'],pat_mean=v['patient_dice_mean'],train_s=m['train_time_s']))
T=pd.DataFrame(rows); display(T); display(T.groupby('cond').agg(['mean','std']).round(4)); T.to_csv(f'{OUT}/val_summary.csv',index=False)